In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F
import country_converter as coco
from itertools import chain

CLEANED_DATA_DIR = Path("../data/cleaned")
PROCESSED_DATA_DIR = Path("../data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{CLEANED_DATA_DIR}/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

df.show(10)

In [ ]:
import torch
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from VehicleAutoencoder import VehicleAutoencoder

def generate_latent_dataframe(
    spark_df, 
    checkpoint_path="best_vehicle_autoencoder.ckpt", 
    scaler_path="scaler_params.pt",
    device="cuda",
    batch_size=65536
) -> pd.DataFrame:
    device_obj = torch.device(device if torch.cuda.is_available() else "cpu")
    model = VehicleAutoencoder.load_from_checkpoint(checkpoint_path).to(device_obj)
    model.eval()

    scaler_params = torch.load(scaler_path, weights_only=False)
    feature_cols = scaler_params["feature_names"]

    motor_col = F.col("Motor energy")
    is_electric = motor_col == "Electricity"
    is_ice = motor_col.isin("Petrol (excluding hybrids)", "Diesel (excluding hybrids)")

    df_zeroed = spark_df.withColumn(
        "co2_emissions_WLTP (g/km)",
        F.when(is_electric, 0.0).otherwise(F.col("co2_emissions_WLTP (g/km)"))
    ).withColumn(
        "engine_capacity (cm3)",
        F.when(is_electric, 0.0).otherwise(F.col("engine_capacity (cm3)"))
    ).withColumn(
        "electric_energy_consumption (Wh/km)",
        F.when(is_ice, 0.0).otherwise(F.col("electric_energy_consumption (Wh/km)"))
    )

    pdf = df_zeroed.dropna(subset=feature_cols).toPandas()
    X_raw = pdf[feature_cols].values.astype(np.float32)
    
    X_scaled = (X_raw - scaler_params["mean"]) / scaler_params["std"]

    latent_coords = []
    with torch.no_grad():
        for i in range(0, len(X_scaled), batch_size):
            batch_x = torch.tensor(X_scaled[i : i + batch_size], dtype=torch.float32, device=device_obj)
            latent_coords.append(model.encode(batch_x).cpu().numpy())

    latent_matrix = np.vstack(latent_coords)
    pdf["z_1"] = latent_matrix[:, 0]
    pdf["z_2"] = latent_matrix[:, 1]

    return pdf

full_enriched_df = generate_latent_dataframe(df)

In [ ]:
full_enriched_df

In [ ]:
import numpy as np
import pandas as pd

def compute_grid_normalisation(
    df: pd.DataFrame, 
    grid_size: float = 0.2, 
    min_registrations: int = 10
) -> pd.DataFrame:
    """
    Computes latent space volume (normalisation factor) and normalized registrations.
    
    Parameters:
    -----------
    df : pd.DataFrame containing ['TIME_PERIOD', 'Motor energy', 'registrations', 'z_1', 'z_2']
    grid_size : Size of each grid cell square in latent z-score units (default: 0.2)
    min_registrations : Threshold to consider a grid cell active (filters out prototype noise)
    
    Returns:
    --------
    pd.DataFrame with normalisation factors and final normalized metrics per group.
    """
    pdf = df.copy()

    # continuous latent space into discrete cell coordinates
    pdf["cell_x"] = np.floor(pdf["z_1"] / grid_size).astype(int)
    pdf["cell_y"] = np.floor(pdf["z_2"] / grid_size).astype(int)

    # aggregate registrations at the grid cell level
    cell_agg = (
        pdf.groupby(["TIME_PERIOD", "Motor energy", "cell_x", "cell_y"])
        .agg(
            cell_registrations=("registrations", "sum"),
            unique_variants=("variant", "nunique") if "variant" in pdf.columns else ("registrations", "count")
        )
        .reset_index()
    )

    # filter noise
    active_cells = cell_agg[cell_agg["cell_registrations"] >= min_registrations]

    # compute the normalisation factor (latent volume = count of occupied active cells)
    volume_df = (
        active_cells.groupby(["TIME_PERIOD", "Motor energy"])
        .agg(
            latent_volume=("cell_x", "count"),
            active_registrations=("cell_registrations", "sum")
        )
        .reset_index()
    )

    # compute total raw registrations per powertrain group
    raw_totals = (
        pdf.groupby(["TIME_PERIOD", "Motor energy"])["registrations"]
        .sum()
        .reset_index()
        .rename(columns={"registrations": "total_raw_registrations"})
    )

    # merge and calculate normalized metric
    summary = pd.merge(volume_df, raw_totals, on=["TIME_PERIOD", "Motor energy"])
    summary["normalized_registrations"] = (
        summary["total_raw_registrations"] / summary["latent_volume"]
    )

    return summary

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

norm_summary = compute_grid_normalisation(
    full_enriched_df, 
    grid_size=0.2,
    min_registrations=10
)

print(norm_summary.head())

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=norm_summary,
    x="TIME_PERIOD",
    y="normalized_registrations",
    hue="Motor energy",
    marker="o",
    linewidth=2.5
)
plt.title("Normalized Registration Density (Sales per Latent Space Unit)", fontweight="bold")
plt.xlabel("Year")
plt.ylabel("Registrations / Active Latent Cell")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()